In [ ]:
import os
import cv2
import numpy as np

def resize_images(folder_path, size=(50, 50)):
    resized_images = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.jpg'):
            img_path = os.path.join(folder_path, filename)
            img = cv2.imread(img_path)
            img_resized = cv2.resize(img, size)
            resized_images.append(img_resized)
    return resized_images

def extract_rgb_channels(images):
    rgb_channels = []
    for img in images:
        channels = cv2.split(img)
        rgb_channels.append(channels)
    return rgb_channels

In [ ]:
import numpy as np

def sturges_bins(data):
    """ Calculate number of bins using Sturges' formula """
    bins = int(np.ceil(1 + np.log2(len(data))))
    return bins

def calculate_histograms(images):
    histograms = []
    for img in images:
        channels = cv2.split(img)
        hist_channels = [cv2.calcHist([channel], [0], None, [256], [0, 256]) for channel in channels]
        histograms.append(hist_channels)
    return histograms

def calculate_relative_frequency(histograms):
    relative_frequencies = []
    for hist_channels in histograms:
        if len(hist_channels) == 3:  # Ensure there are three channels
            total_pixels = sum([h.sum() for h in hist_channels])  # Total pixels across all channels
            rel_freq_channels = [(hist / total_pixels).flatten() for hist in hist_channels]  # Flatten and normalize
            relative_frequencies.append(rel_freq_channels)
        else:
            print("Histograms do not have three channels. Check the input.")
            return None
    return relative_frequencies


In [ ]:
import numpy as np

def calculate_dependent_joint_rgb_frequency(images):
    joint_frequencies = []
    for img in images:
        # Create an empty frequency matrix for RGB combinations
        joint_frequency = np.zeros((256, 256, 256))
        for row in img:
            for pixel in row:
                r, g, b = pixel
                joint_frequency[r, g, b] += 1
        # Normalize the frequencies to get relative frequency
        joint_frequency /= joint_frequency.sum()
        joint_frequencies.append(joint_frequency)
    return joint_frequencies



In [ ]:
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import pairwise_distances_argmin
import numpy as np
import cv2

def load_and_preprocess_images(folder_path, filenames):
    images = []
    for filename in filenames:
        img_path = os.path.join(folder_path, filename)
        img = cv2.imread(img_path)
        img_resized = cv2.resize(img, (50, 50))
        img_flattened = img_resized.flatten()
        images.append(img_flattened)
    return np.array(images)

def js_divergence_centroid(X, centroids):
    """ Compute JS divergence from each point to each centroid """
    js_distances = np.zeros((X.shape[0], centroids.shape[0]))
    for i, centroid in enumerate(centroids):
        for j, x in enumerate(X):
            js_distances[j, i] = jensenshannon(x, centroid)
    return js_distances

def k_means_js(X, initial_centroids, max_iter=100):
    centroids = initial_centroids
    for _ in range(max_iter):
        js_distances = js_divergence_centroid(X, centroids)
        labels = np.argmin(js_distances, axis=1)
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(centroids.shape[0])])
        if np.all(centroids == new_centroids):
            break
        centroids = new_centroids
    return labels, centroids




Hasilnya

In [ ]:
import os

def k_means_js_with_filenames(folder_path, initial_centroids, max_iter=100):
    # Load and preprocess all images in the folder
    filenames = []
    images = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.jpg'):
            img_path = os.path.join(folder_path, filename)
            img = cv2.imread(img_path)
            img_resized = cv2.resize(img, (50, 50))
            img_flattened = img_resized.flatten()
            images.append(img_flattened)
            filenames.append(filename)
    images = np.array(images)

    # Perform K-means clustering
    clusters, final_centroids = k_means_js(images, initial_centroids, max_iter)

    # Map filenames to their clusters
    filename_cluster_map = list(zip(filenames, clusters))
    return filename_cluster_map

# Example Usage
folder_path = '/content/drive/MyDrive/Colab Notebooks/bunga'
initial_files = ['O1.jpg', 'S1.jpg']
initial_centroids = load_and_preprocess_images(folder_path, initial_files)

filename_cluster_map = k_means_js_with_filenames(folder_path, initial_centroids)

# Print the filenames along with their cluster groups
for filename, cluster in filename_cluster_map:
    print(f"{filename}: Cluster {cluster}")


O2.jpg: Cluster 0
O6.jpg: Cluster 0
S2.jpg: Cluster 0
O7.jpg: Cluster 0
S6.jpg: Cluster 1
O9.jpg: Cluster 1
S10.jpg: Cluster 1
S7.jpg: Cluster 1
S1.jpg: Cluster 1
S5.jpg: Cluster 1
S4.jpg: Cluster 0
O8.jpg: Cluster 1
S3.jpg: Cluster 1
O5.jpg: Cluster 0
O3.jpg: Cluster 0
S9.jpg: Cluster 1
O10.jpg: Cluster 1
O1.jpg: Cluster 0
O4.jpg: Cluster 1
S8.jpg: Cluster 1


In [ ]:
import numpy as np
import cv2
import os

# Assuming the function calculate_dependent_joint_rgb_frequency is defined as before
def calculate_dependent_joint_rgb_frequency(images):
    joint_frequencies = []
    for img in images:
        # Create an empty frequency matrix for RGB combinations
        joint_frequency = np.zeros((256, 256, 256))
        for row in img:
            for pixel in row:
                r, g, b = pixel
                joint_frequency[r, g, b] += 1
        # Normalize the frequencies to get relative frequency
        joint_frequency /= joint_frequency.sum()
        joint_frequencies.append(joint_frequency)
    return joint_frequencies

def load_images(folder_path):
    images = []
    filenames = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.jpg'):
            img_path = os.path.join(folder_path, filename)
            img = cv2.imread(img_path)
            img_resized = cv2.resize(img, (50, 50))
            images.append(img_resized)
            filenames.append(filename)
    return images, filenames

# Load and process images from a folder
folder_path = '/content/drive/MyDrive/Colab Notebooks/bunga'
images, filenames = load_images(folder_path)

# Calculate the joint RGB frequency for each image
joint_rgb_frequencies = calculate_dependent_joint_rgb_frequency(images)

# Output joint RGB frequency data
for filename, freq in zip(filenames, joint_rgb_frequencies):
    print(f"Filename: {filename}, Joint RGB Frequency Data:\n{freq}")

# You can now inspect the joint RGB frequency data for each image


Filename: O2.jpg, Joint RGB Frequency Data:
[[[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 ...

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0

In [ ]:
import pandas as pd
import numpy as np
import cv2
import os

def save_to_excel(folder_path, excel_path):
    images, filenames = load_images(folder_path)
    histograms = calculate_histograms(images)
    relative_frequencies = calculate_relative_frequency(histograms)

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        for filename, rel_freq in zip(filenames, relative_frequencies):
            try:
                if len(rel_freq) == 3:
                    df_rel_freq = pd.DataFrame({'R_Freq': rel_freq[0],
                                                'G_Freq': rel_freq[1],
                                                'B_Freq': rel_freq[2]})
                    df_rel_freq.to_excel(writer, sheet_name=f'{filename}_RelFreq', index=False)
                else:
                    print(f"Skipping {filename}: Unexpected relative frequency data shape.")
            except Exception as e:
                print(f"Error processing {filename}: {e}")

    print(f"Data saved to {excel_path}")

# Example Usage
histograms = calculate_histograms(images)
for hist in histograms:
    print(f"Histogram shape: {[h.shape for h in hist]}")  # Debugging statement

folder_path = '/content/drive/MyDrive/Colab Notebooks/bunga'
excel_path = '/content/drive/MyDrive/Colab Notebooks/kumpulan excel/output_frequencies.xlsx'
save_to_excel(folder_path, excel_path)

Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Histogram shape: [(256, 1), (256, 1), (256, 1)]
Data saved to /content/drive/MyDrive/Col

In [ ]:
import pandas as pd
import numpy as np
import cv2
import os
import math

def calculate_sturges_bins(image):
    num_pixels = image.shape[0] * image.shape[1]
    bins = int(math.ceil(1 + np.log2(num_pixels)))
    return bins

def calculate_histograms(images):
    histograms = []
    for img in images:
        bins = calculate_sturges_bins(img)
        hist_channels = [cv2.calcHist([ch], [0], None, [bins], [0, 256]).flatten() for ch in cv2.split(img)]
        histograms.append(hist_channels)
    return histograms

def calculate_relative_frequency(histograms):
    relative_frequencies = []
    for hist_channels in histograms:
        total_pixels = sum([h.sum() for h in hist_channels])
        rel_freq_channels = [(hist / total_pixels) for hist in hist_channels]
        relative_frequencies.append(rel_freq_channels)
    return relative_frequencies

def calculate_dependent_joint_rgb_frequency(images):
    joint_frequencies = []
    for img in images:
        joint_frequency = np.zeros((256, 256, 256))
        for row in img:
            for pixel in row:
                r, g, b = pixel
                joint_frequency[r, g, b] += 1
        joint_frequency /= joint_frequency.sum()  # Normalize to get relative frequency
        joint_frequencies.append(joint_frequency)
    return joint_frequencies

def save_to_excel(folder_path, excel_path):
    images, filenames = load_images(folder_path)
    histograms = calculate_histograms(images)
    relative_frequencies = calculate_relative_frequency(histograms)
    joint_rgb_frequencies = calculate_dependent_joint_rgb_frequency(images)

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        for filename, rel_freq, joint_freq in zip(filenames, relative_frequencies, joint_rgb_frequencies):
            # Save Relative Frequency
            df_rel_freq = pd.DataFrame({
                'R_Freq': rel_freq[0],
                'G_Freq': rel_freq[1],
                'B_Freq': rel_freq[2]
            })
            df_rel_freq.to_excel(writer, sheet_name=f'{filename}_RelFreq', index=False)

            # Summarize and Save Joint RGB Frequency
            joint_freq_summary = {
                'R_avg': np.mean(joint_freq, axis=(1, 2)),
                'G_avg': np.mean(joint_freq, axis=(0, 2)),
                'B_avg': np.mean(joint_freq, axis=(0, 1))
            }
            df_joint_freq_summary = pd.DataFrame(joint_freq_summary)
            df_joint_freq_summary.to_excel(writer, sheet_name=f'{filename}_JointRGB', index=False)

    print(f"Data saved to {excel_path}")



# Example Usage
folder_path = '/content/drive/MyDrive/Colab Notebooks/bunga'
excel_path = '/content/drive/MyDrive/Colab Notebooks/kumpulan excel/output_frequencies.xlsx'
save_to_excel(folder_path, excel_path)


Data saved to /content/drive/MyDrive/Colab Notebooks/kumpulan excel/output_frequencies.xlsx


In [ ]:
import numpy as np

# Relative Frequencies for each bin in each channel
rel_freq_R = [0.4, 0.6]  # Red channel
rel_freq_G = [0.5, 0.5]  # Green channel
rel_freq_B = [0.7, 0.3]  # Blue channel

# Simulating a non-independent joint frequency distribution
joint_rgb_freq = np.zeros((2, 2, 2))  # 2 bins for each of R, G, B

# Hypothetical dependency pattern
# For simplicity, let's assume higher values in R increase the probability of higher values in G and B
for r in range(2):
    for g in range(2):
        for b in range(2):
            joint_rgb_freq[r, g, b] = rel_freq_R[r] * (rel_freq_G[g] + 0.1 * r) * (rel_freq_B[b] + 0.1 * r)

# Normalizing to ensure the sum is 1
joint_rgb_freq /= joint_rgb_freq.sum()

print("Joint RGB Frequencies (Non-Independent):")
print(joint_rgb_freq)


Joint RGB Frequencies (Non-Independent):
[[[0.11075949 0.04746835]
  [0.11075949 0.04746835]]

 [[0.2278481  0.11392405]
  [0.2278481  0.11392405]]]
